In [ ]:
!git clone https://github.com/ajurcik/MLPrague-2026-test.git
!pip install ./MLPrague-2026-test
!pip install torch-geometric
!cp -a MLPrague-2026-test/images .

# Supervised Anomaly Detection on Graphs — Hands-on Workshop

In this notebook you will progressively integrate graph structure into **supervised** anomaly detection task on the **YelpChi** fraud-review dataset.

**Approaches we will train & compare:**
- **Manual graph features + Random Forest classifier** — compute topological features, combine with tabular features
- **Node2Vec embeddings + Random Forest classifier** — learn node representations from graph structure
- **Graph Convolutional Network** — end-to-end learning
- Bonus: **BWGNN** — spectral GNN specialized in fraud detection

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from IPython.display import Markdown, display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_recall_curve, classification_report, f1_score

from ml_prague_2026.evaluation import compare_models, evaluate_model
from ml_prague_2026.losses import FocalLoss, calculate_class_weights
from ml_prague_2026.utils import stratified_node_split
from ml_prague_2026.models import train_chart

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from torch_geometric.seed import seed_everything

seed_everything(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

## 1. Data loading & graph construction

The **YelpChi** dataset contains 46k Yelp reviews of Chicago businesses (like restaurants etc.) with hand-crafted features and a binary `spam` label (see the introduction notebook for full exploration).

Reviews are connected through three types of relations:
- **R-U-R** (Review-User-Review): two reviews written by the **same user** (~49k edges)
- **R-S-R** (Review-Star-Review): two reviews of the **same business** with the **same star rating** (~3.4M edges)
- **R-T-R** (Review-Time-Review): two reviews of the **same business** within a **similar time period** (~574k edges)

In [ ]:
# Dataset paths
BASE_PATH = 'MLPrague-2026-test/'

YELP_CHI_PATH = BASE_PATH + 'data/yelpchi.parquet'

YELP_CHI_EDGE_RUR_PATH = BASE_PATH + 'data/yelpchi_rur.npy'
YELP_CHI_EDGE_RSR_PATH = BASE_PATH + 'data/yelpchi_rsr.npy'
YELP_CHI_EDGE_RTR_PATH = BASE_PATH + 'data/yelpchi_rtr.npy'

# Load the dataset
yelp_chi = pd.read_parquet(YELP_CHI_PATH)
FEATURES = [c for c in yelp_chi.columns if c.startswith('f_')]

display(yelp_chi.head(3))
print()
print(f'Reviews: {len(yelp_chi):,}, Features: {len(FEATURES)}, Spam ratio: {yelp_chi["spam"].mean():.1%}')

### TASK 1.A: Build the R-U-R (Review-User-Review) edge list

If multiple reviews were written by the same user, that's a useful signal — a fraudulent user likely writes multiple fake reviews. We capture this by **connecting reviews that share an author** with an R-U-R (Review-User-Review) edge.

Build an edge list — a numpy array of shape `(num_edges, 2)` — containing pairs of `review_idx`s that share the same `user_id`.

In [ ]:
# TASK 1.A: build R-U-R edges from user_id
# - each row in yelp_chi has a 'review_id' and 'user_id'
# - connect all pairs of reviews that share the same user_id
# - result should be a numpy array of shape (num_edges, 2)
#
# - Hint: Use a self-merge of reviews on `user_id`, then filter out self-loops (a review paired with itself)
#         and duplicate edges (those in the opposite direction, if any), e.g. using comparison on their review_idx columns.

edges_rur = None  ### YOUR CODE HERE ###

<!-- #region -->
<details>
<summary><b>Task 1.A — Solution</b></summary>

```python
edges_rur = (
    yelp_chi[['review_id', 'user_id']]
        .merge(yelp_chi[['review_id', 'user_id']], on='user_id')
        .query('review_id_x < review_id_y')
        [['review_id_x', 'review_id_y']]
        .values
)
```

</details>
<!-- #endregion -->

In [ ]:
print(f'R-U-R edges built: {len(edges_rur):,}')

# Load pre-computed edge lists for the other two relation types
edges_rsr = np.load(YELP_CHI_EDGE_RSR_PATH, allow_pickle=True)
edges_rtr = np.load(YELP_CHI_EDGE_RTR_PATH, allow_pickle=True)
print(f'R-S-R edges loaded: {len(edges_rsr):,}')
print(f'R-T-R edges loaded: {len(edges_rtr):,}')

In [ ]:
# Combine all three edge types into one edge list
edges_all = np.concatenate([edges_rur, edges_rsr, edges_rtr], axis=0)

# Undirected graph: add edges in both directions
src = np.array([s for s in edges_all[:, 0]])
dst = np.array([d for d in edges_all[:, 1]])

edge_index = torch.tensor(
    np.stack([np.concatenate([src, dst]), np.concatenate([dst, src])]),
    dtype=torch.long,
).contiguous()

# Node features and labels
x = torch.tensor(yelp_chi[FEATURES].values, dtype=torch.float)
y = torch.tensor(yelp_chi['spam'].values, dtype=torch.long)

# Create pytorch geometric `Data` object with stratified train/val/test split
data = Data(x=x, edge_index=edge_index, y=y)
data = stratified_node_split(data, train_ratio=0.7, val_ratio=0.1)
data = data.to(device)
data

In [ ]:
X_tab = yelp_chi[FEATURES].values
y_all = yelp_chi['spam'].values

idx_train = data.train_mask.cpu().numpy().nonzero()[0]
idx_val = data.val_mask.cpu().numpy().nonzero()[0]
idx_test = data.test_mask.cpu().numpy().nonzero()[0]

X_train, X_val, X_test = X_tab[idx_train], X_tab[idx_val], X_tab[idx_test]
y_train, y_val, y_test = y_all[idx_train], y_all[idx_val], y_all[idx_test]
print(f'Train: {len(X_train):,}, Val: {len(X_val):,}, Test: {len(X_test):,}')

np.savez(BASE_PATH + 'data/yelpchi_split.npz', train=idx_train, val=idx_val, test=idx_test)

### Random Forest Baseline
Train Random Forest on tabular features only - no graph information

In [ ]:
def randomForest(X_train, y_train, X_test, random_state=42):
    """Trains RandomForest classifier on X_train and then predicts X_test,
    returning predictions and probabilities separately."""
    rf = RandomForestClassifier(random_state=random_state)
    rf.fit(X_train, y_train)
    proba = rf.predict_proba(X_test)
    preds = rf.classes_[proba.argmax(axis=1)]
    return preds, proba[:, 1]

In [ ]:
# Train Random Forest on tabular features only
_preds, _probs = randomForest(X_train, y_train, X_test)
baseline_results = evaluate_model('RF (tabular)', y_test, _preds, _probs)

## 2. Manual graph features + ML classifier

The simplest way to leverage graph structure is to compute **topological features** for each node and add them as extra columns alongside the existing 32 features.

The most basic graph feature is **degree** — the number of edges connected to a node. Since we have three edge types, we can compute a separate degree for each, capturing how connected a review is through user, rating, and temporal relations.

### TASK 2.A: Compute degree features

The **degree** of a node is the number of edges connected to it. Since we have three edge types, compute a separate degree for each.

In [ ]:
# TASK 2.A: Complete function computing node degree and apply it to each edge type
# - edges_rur, edges_rsr, edges_rtr each have shape (num_edges, 2) with numerical review_idx
# - compute degree_rur, degree_rsr, degree_rtr as arrays of length num_nodes
#
# - Hint: Count how often each node index appears in the edge array. Each edge contributes to the degree of both its endpoints.
#         There are many possible ways — one is using value_counts() from pandas and then reindex to range of numerical indices.

def compute_degree(edges, num_nodes):
    ### YOUR CODE HERE ###
    return None

degree_rur = compute_degree(edges_rur, data.num_nodes)
degree_rsr = compute_degree(edges_rsr, data.num_nodes)
degree_rtr = compute_degree(edges_rtr, data.num_nodes)

<!-- #region -->
<details>
<summary><b>Task 2.A — Solution</b></summary>

There are many possible solutions. Here is one using pandas:

```python
def compute_degree(edges, num_nodes):
    node_ids = [node for edge in edges for node in edge]
    return (pd.Series(node_ids)
            .value_counts()
            .reindex(range(num_nodes), fill_value=0)
            .values)
```

</details>
<!-- #endregion -->

In [ ]:
# Degree distributions by class (clipped at 99th percentile for readability)
fig, axes = plt.subplots(1, 3, figsize=(14, 3))
for ax, deg, name in zip(axes, [degree_rur, degree_rsr, degree_rtr], ['R-U-R', 'R-S-R', 'R-T-R']):
    q99 = np.quantile(deg, 0.99)
    bins = np.linspace(0, q99, 51)
    ax.hist(deg[y_all == 0].clip(max=q99), bins=bins, density=True, color='steelblue', label='normal')
    ax.hist(deg[y_all == 1].clip(max=q99), bins=bins, density=True, color='salmon', label='spam', alpha=0.7)
    ax.set_title(f'{name} degree')
    ax.legend()
plt.tight_layout()
plt.show();

### Random Forest with Degree Features
Train Random Forest with degree features only - no tabular data.

In [ ]:
# Use only degree features
X_degree = np.column_stack([degree_rur, degree_rsr, degree_rtr])

# RF with degree features only
_preds, _probs = randomForest(X_degree[idx_train], y_train, X_degree[idx_test])
degree_results = evaluate_model('RF (degree)', y_test, _preds, _probs)

### Random Forest with Degree Features and Tabular Data
Combine degree with tabular features.

In [ ]:
# Combine tabular features with degree features
X_train_deg_comb = np.concatenate([X_train, X_degree[idx_train]], axis=1)
X_test_deg_comb = np.concatenate([X_test, X_degree[idx_test]], axis=1)

# RF with tabular + degree features
_preds, _probs = randomForest(X_train_deg_comb, y_train, X_test_deg_comb)
degree_results_comb = evaluate_model('RF (tabular + degree)', y_test, _preds, _probs)

In [ ]:
compare_models([baseline_results, degree_results, degree_results_comb]);

## 3. Node2Vec + ML classifier

Instead of hand-crafting graph features, we can **learn** node representations automatically.

[Node2Vec](https://arxiv.org/abs/1607.00653) performs biased random walks on the graph and uses a Skip-gram model (similar to Word2Vec) to learn embeddings that capture both local and global graph structure.

**How Node2Vec works:**
1. Perform random walks starting from each node (controlled by parameters `p` and `q`)
2. Treat walks as "sentences" and nodes as "words"
3. Apply Word2Vec (Skip-gram) to learn node embeddings

<center><img src="images/node2vec.png" width="600"></center>

&nbsp;

Parameters `p` (return) and `q` (in-out) control the walk behavior:
- Low `q` → BFS-like walks → capture local structure
- Low `p` → DFS-like walks → capture community structure

In [ ]:
# Kept for reference only, we will actually use precomputed embeddings due to time constraints, see the next cell.

# karateclub installation for Node2Vec.
#
#! pip install karateclub==1.2.0 --force-reinstall --no-cache-dir

# Build NetworkX graph for Node2Vec (requires undirected NetworkX graph)
#
# import networkx as nx
# from karateclub import Node2Vec
#
# G = nx.Graph()
# G.add_nodes_from(range(num_nodes))
# G.add_edges_from(zip(src, dst))
# print(f'Graph: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges')

# Fit Node2Vec: random walks + Skip-gram to learn 32-dim node embeddings
#
# node2vec = Node2Vec(walk_length=40, walk_number=40, dimensions=32, p=0.25, q=0.5, workers=8, seed=42)
# node2vec.fit(G)
# node2vec_embeddings = node2vec.get_embedding()
# np.save(BASE_PATH + 'data/node2vec_embeddings_2.npy', node2vec_embeddings)

In [ ]:
node2vec_embeddings = np.load(BASE_PATH + 'data/node2vec_embeddings.npy')

### Random Forest with Node2Vec Embeddings

In [ ]:
# Train RF on Node2Vec embeddings only (no tabular features)
X_emb_train = node2vec_embeddings[idx_train]
X_emb_test = node2vec_embeddings[idx_test]

# RF with embeddings
_preds, _probs = randomForest(X_emb_train, y_train, X_emb_test)
emb_results = evaluate_model('RF (embedding)', y_test, _preds, _probs)

### Random Forest with Node2Vec Embeddings and Tabular Features

In [ ]:
# Node2Vec embeddings + tabular features combined
X_comb_train = np.concatenate([X_train, X_emb_train], axis=1)
X_comb_test = np.concatenate([X_test, X_emb_test], axis=1)

# RF with embeddings
_preds, _probs = randomForest(X_comb_train, y_train, X_comb_test)
combined_results = evaluate_model('RF (tabular + embedding)', y_test, _preds, _probs)

In [ ]:
compare_models([baseline_results, degree_results, degree_results_comb, emb_results, combined_results]);

## 4. Graph Convolutional Network

**Graph Convolutional Network (GCN)** is the simplest GNN architecture. Each layer aggregates features from all neighbors with learnable weights, then applies a nonlinearity.

After stacking multiple layers, each node's representation encodes information from its multi-hop neighborhood. The model learns both the features and the classifier **end-to-end** — directly optimizing for the classification task.

### TASK 4.A: Define the GCN convolutional layers

<img src="images/gcn-vis.png">

Complete the `__init__` method of the `GCN` class by creating `self.convs` — a [ModuleList](https://docs.pytorch.org/docs/stable/generated/torch.nn.ModuleList.html) of `num_layers` [SAGEConv](https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.nn.conv.SAGEConv.html) layers.

The output projection head and the `forward` method are already provided.

In [ ]:
class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels=64, out_channels=2, num_layers=3, dropout=0.08):
        super().__init__()
        self.dropout_rate = dropout

        # TASK 4.A: Define the GCN convolutional layers
        # - create self.convs as a ModuleList of num_layers SAGEConv layers
        # - the first layer maps in_channels → hidden_channels
        # - the remaining layers map hidden_channels → hidden_channels
        # - (where the input_channels is size of our input samples, i.e. the number of features)
        #
        # - Hint: Create an empty ModuleList, append the first layer, then append the rest in a loop.
        #
        ### YOUR CODE HERE ###
        #
        self.convs = None  ### MODIFY THIS ###
        #
        ### YOUR CODE HERE ###

        self.head = nn.Linear(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        for conv in self.convs:
            x = conv(x, edge_index)
            x = F.mish(x)
            x = F.dropout(x, p=self.dropout_rate, training=self.training)
        return self.head(x)

<!-- #region -->
<details>
<summary><b>Task 4.A — Solution</b></summary>

```python
self.convs = torch.nn.ModuleList()
self.convs.append(SAGEConv(in_channels, hidden_channels))
for _ in range(num_layers - 1):
    self.convs.append(SAGEConv(hidden_channels, hidden_channels))
```

</details>
<!-- #endregion -->

In [ ]:
def train_gnn(model, data, criterion, epochs=500, lr=0.0032, log_interval=50):
    """Train a GNN model with given loss function (the `criterion` parameter). Returns training history."""
    seed_everything(42)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {'train_loss': [], 'val_loss': [], 'train_f1': [], 'val_f1': []}

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = criterion(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            out = model(data.x, data.edge_index)
            train_loss = criterion(out[data.train_mask], data.y[data.train_mask])
            val_loss = criterion(out[data.val_mask], data.y[data.val_mask])
            pred = out.argmax(dim=1)
            train_f1 = f1_score(data.y[data.train_mask].cpu(), pred[data.train_mask].cpu(), average='macro')
            val_f1 = f1_score(data.y[data.val_mask].cpu(), pred[data.val_mask].cpu(), average='macro')

        history['train_loss'].append(train_loss.item())
        history['val_loss'].append(val_loss.item())
        history['train_f1'].append(train_f1)
        history['val_f1'].append(val_f1)

        if epoch == 1 or epoch % log_interval == 0:
            print(f'Epoch {epoch:03d} | Train loss: {loss.item():.4f} | Val Loss: {val_loss.item():.4f} | Train F1: {train_f1:.4f} | Val F1: {val_f1:.4f}')

    return history


def evaluate_gnn(name, model, data):
    """Evaluate a trained GNN on the test mask."""
    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(data.x, data.edge_index), dim=1)[:, 1]
    mask = data.test_mask
    return evaluate_model(name, data.y[mask].cpu().numpy(), (probs[mask] > 0.5).int().cpu().numpy(), probs[mask].cpu().numpy());

### Compare loss functions for handling class imbalance

Class imbalance (14.5% spam vs 85.5% normal) is a key challenge for training GNNs. Without any adjustments, the model may learn to simply predict the majority class.

Try three approaches:
1. **No weighting** — standard `CrossEntropyLoss`
2. **Class weights** — weight the minority class higher in `CrossEntropyLoss`
3. **Focal Loss** — down-weight easy examples, focus on hard ones (using `FocalLoss` from `ml_prague_2026.losses`)

Run each variant below and compare the results.

In [ ]:
# Variant 1: no weighting - val_loss computed in train mode
gcn_v1 = GCN(data.num_features).to(device)
criterion_v1 = torch.nn.CrossEntropyLoss()
history_v1 = train_gnn(gcn_v1, data, criterion_v1)
train_chart(history_v1)

In [ ]:
# Variant 2: class weights
class_weights = calculate_class_weights(data)
print(f'Class weights: {class_weights}')

In [ ]:
gcn_v2 = GCN(data.num_features).to(device)
criterion_v2 = torch.nn.CrossEntropyLoss(weight=class_weights)
history_v2 = train_gnn(gcn_v2, data, criterion_v2)
train_chart(history_v2)

In [ ]:
# Variant 3: focal loss
gcn_v3 = GCN(data.num_features).to(device)
criterion_v3 = FocalLoss(alpha=class_weights*1.45, gamma=1.4)
history_v3 = train_gnn(gcn_v3, data, criterion_v3)
train_chart(history_v3)

In [ ]:
# Evaluate all three GCN loss variants on test set
display(Markdown('**GCN (no weights)**'))
gcn_v1_results = evaluate_gnn('GCN (no weights)', gcn_v1, data);

display(Markdown('**GCN (class weights)**'))
gcn_v2_results = evaluate_gnn('GCN (class weights)', gcn_v2, data);

display(Markdown('**GCN (focal loss)**'))
gcn_v3_results = evaluate_gnn('GCN (focal loss)', gcn_v3, data);

In [ ]:
compare_models([gcn_v1_results, gcn_v2_results, gcn_v3_results]);

## 5. Final comparison

Let's compare all approaches side-by-side.

In [ ]:
compare_models([
    baseline_results,
    degree_results,
    degree_results_comb,
    emb_results,
    combined_results,
    gcn_v3_results
]);

### Classification examples

In [ ]:
gcn_v3.eval()
with torch.no_grad():
    gcn_v3_preds = torch.softmax(gcn_v3(data.x, data.edge_index), dim=1)[:, 1]
test_examples = yelp_chi[(gcn_v3_preds > 0.5).cpu().numpy() & data.test_mask.cpu().numpy()][["review","spam"]]

In [ ]:
for i, r in test_examples.sample(5).iterrows():
    display(Markdown(f"#### Classified Spam Example #{i}"))
    display(Markdown(f"- Is real spam? **{r['spam'] and 'Yes' or 'No'}**"))
    display(Markdown(f"> {r['review']}"))

### Takeaway Summary

YelpChi has strong hand-crafted features, so the tabular baselines are already very competitive. In real-world scenarios with fewer or weaker features, graph-based approaches typically provide much larger improvements.

| Approach | Graph integration | Pros | Cons |
|----------|-------------------|------|------|
| Graph features + RF | Manual feature engineering | Simple, interpretable, works with any classifier | Requires domain expertise, may not capture all structure |
| Node2Vec + RF | Automatic embeddings | No manual feature engineering, captures walk-based patterns | Two-stage pipeline, embeddings not optimized for the task |
| GCN | End-to-end learning | Optimizes features + classifier together | Slower and more complex |

## 6. Threshold tuning (optional)

The default classification threshold of 0.5 is often suboptimal. The **precision-recall curve** shows the tradeoff between catching more anomalies (higher recall) and making fewer false alarms (higher precision). 

<img src="images/precision-recall-curve.png" width="600">

The "best" threshold depends on the chosen metric — which is typically only a proxy for the actual business objective (e.g. cost of missed fraud vs. cost of false alarms).

In [ ]:
# Get NN predicted probabilities on test set
gcn_v3.eval()
with torch.no_grad():
    probs_gcn = torch.softmax(gcn_v3(data.x, data.edge_index), dim=1)[:, 1]

# Plot precision-recall curve
precision, recall, thresholds = precision_recall_curve(data.y[data.test_mask].cpu().numpy(), probs_gcn[data.test_mask].cpu().numpy())

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(recall, precision)
ax.set(xlabel='Recall', ylabel='Precision', title='Precision-recall curve')
plt.tight_layout()
plt.show();

### TASK 6.A: Tune the classification threshold

In practice, the business goal determines the right tradeoff between precision and recall.

Try different classification thresholds and observe how precision/recall change:
- **Conservative** (high threshold, e.g. 0.8): fewer false positives, misses more spam
- **Aggressive** (low threshold, e.g. 0.3): catches more spam, more false alarms

**Goal**: Find a threshold that achieves at least 70% precision on the spam class.

In [ ]:
# TASK 6.A: Try different classification thresholds
# Change the threshold below and re-run to see the effect.

threshold = 0.5  ### MODIFY THIS ###

preds_tuned = (probs_gcn[data.test_mask] > threshold).int().cpu().numpy()
print(classification_report(data.y[data.test_mask].cpu().numpy(), preds_tuned, target_names=['normal', 'spam']))

## 7. BWGNN (optional)

[BWGNN](https://proceedings.mlr.press/v162/tang22b.html) (Beta Wavelet Graph Neural Network) is a spectral GNN designed for anomaly detection. Its **key insight is that anomalies shift graph signal energy toward higher frequencies**, which standard GNNs (being low-pass filters) miss. BWGNN uses a collection of Beta wavelet filters, each tuned to a different frequency band, to capture this anomaly signal across the full spectrum.

This is an advanced model — just run the cells below and compare with our previous results.

In [ ]:
from ml_prague_2026.models import SupervisedBWGNN

class_weights = calculate_class_weights(data)

bwgnn = SupervisedBWGNN(
    in_feats=data.num_features, h_feats=64, num_classes=2,
    edge_index=data.edge_index, num_nodes=data.num_nodes,
    d=2, lr=0.007, alpha=class_weights, gamma=2.0, device=device, dropout=0.2
)

bwgnn.fit(data.x, data.y, data.train_mask, data.val_mask, epochs=600, log_interval=50)

In [ ]:
probs_bwgnn = bwgnn.predict_proba(data.x)[:, 1][data.test_mask].cpu().detach().numpy()
preds_bwgnn = (probs_bwgnn > 0.5).astype(int)
y_test_gnn = data.y[data.test_mask].cpu().numpy()

bwgnn_results = evaluate_model('BWGNN', y_test_gnn, preds_bwgnn, probs_bwgnn)

In [ ]:
compare_models([
    baseline_results,
    degree_results,
    degree_results_comb,
    emb_results,
    combined_results,
    gcn_v3_results,
    bwgnn_results
]);